In [ ]:
# Instalar dependencias del proyecto antes de ejecutar el notebook
%pip install -r ../requirements.txt

Usaremos como entrada los artefactos corregidos generados por el pipeline:

- `IA_Proyecto/data/processed/fintech_limpio_*.csv`
- `IA_Proyecto/data/processed/fintech_invalidos_*.csv`
- `IA_Proyecto/data/kpi/kpi_fintech_*.csv`

Cargaremos en PostgreSQL:

- `ventas_clean`
- `ventas_error`
- `kpi_fintech`

Instalación sugerida:

```bash
pip install pandas sqlalchemy psycopg2-binary python-dotenv
```

# Actividad 2.4 — Carga de datos a Base de Datos

## Objetivo

En este notebook implementaremos la etapa final del pipeline de datos: la carga controlada a una base de datos relacional PostgreSQL.

El flujo completo trabajado hasta ahora es:

1. Ingesta de datos
2. Limpieza y transformación
3. Validación estructural y semántica
4. Carga a base de datos ← etapa actual

En esta etapa cargaremos los registros válidos en la tabla `ventas_clean` y los registros rechazados en la tabla `ventas_error`.

## Importancia de la carga controlada

La carga de datos no consiste solo en insertar registros en una tabla. También implica:

- validar conexión a la base de datos
- verificar estructura de tablas
- respetar tipos de datos
- manejar errores
- registrar trazabilidad
- separar datos insertados y rechazados

Esto permite que el pipeline sea reproducible, auditable y confiable.

In [ ]:
from pathlib import Path
import logging
import os

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from sqlalchemy import create_engine, text

from IA_Proyecto.src.audit_utils import latest_version_path

print("Bibliotecas cargadas...")

In [ ]:
os.makedirs("../logs", exist_ok=True)

logging.basicConfig(
    filename="../logs/load_database.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Inicio del proceso de carga a base de datos")
print("Logger configurado correctamente")

## Conexión a PostgreSQL

La conexión se realizará usando variables de entorno desde el archivo `.env`.

Esto evita dejar credenciales directamente escritas en el notebook.

DB_HOST=TU_HOST

DB_PORT=TU_PORT

DB_NAME=TU_NAME

DB_USER=TU_USUARIO

DB_PASSWORD=TU_PASSWORD

In [ ]:
load_dotenv("../.env")

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

missing_env = [name for name, value in {
    "DB_HOST": DB_HOST,
    "DB_PORT": DB_PORT,
    "DB_NAME": DB_NAME,
    "DB_USER": DB_USER,
    "DB_PASSWORD": DB_PASSWORD,
}.items() if not value]

if missing_env:
    print(f"Variables faltantes en .env: {', '.join(missing_env)}")

connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

print("Cadena de conexión creada correctamente")

In [ ]:
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT current_database(), current_schema();"))
        print(result.fetchone())
        logging.info("Conexión a PostgreSQL exitosa")
except Exception as e:
    logging.error(f"Error de conexión a PostgreSQL: {e}")
    print("Error de conexión:", e)

## Lectura de artefactos corregidos

A continuación tomaremos la última versión generada por el pipeline para:

- registros válidos corregidos
- registros rechazados
- KPI de monitoreo

In [ ]:
base_data_dir = Path("../data").resolve()
processed_dir = base_data_dir / "processed"
kpi_dir = base_data_dir / "kpi"

ruta_validos = latest_version_path(processed_dir, "fintech_limpio")
ruta_invalidos = latest_version_path(processed_dir, "fintech_invalidos")
ruta_kpi = latest_version_path(kpi_dir, "kpi_fintech")

if ruta_validos is None:
    raise FileNotFoundError(f"No se encontró un archivo de válidos en {processed_dir}")
if ruta_invalidos is None:
    raise FileNotFoundError(f"No se encontró un archivo de inválidos en {processed_dir}")
if ruta_kpi is None:
    raise FileNotFoundError(f"No se encontró un archivo de KPI en {kpi_dir}")

print("Archivo válidos:", ruta_validos.name)
print("Archivo inválidos:", ruta_invalidos.name)
print("Archivo KPI:", ruta_kpi.name)

df_validos = pd.read_csv(ruta_validos)
df_invalidos = pd.read_csv(ruta_invalidos)
df_kpi = pd.read_csv(ruta_kpi)

print("Registros válidos:", len(df_validos))
print("Registros inválidos:", len(df_invalidos))
print("Registros KPI:", len(df_kpi))

## Vista previa de los datos

Revisamos una muestra de cada archivo antes de cargarla en la base de datos.

In [ ]:
print("Muestra de registros válidos")
display(df_validos.head())

print("Muestra de registros inválidos")
display(df_invalidos.head())

print("Muestra del KPI")
display(df_kpi.head())

## Carga en PostgreSQL

Los registros válidos se cargarán en `ventas_clean`, los rechazados en `ventas_error` y el KPI en `kpi_fintech`.

In [ ]:
tablas_carga = {
    "ventas_clean": df_validos,
    "ventas_error": df_invalidos,
    "kpi_fintech": df_kpi,
}

for nombre_tabla, dataframe in tablas_carga.items():
    if not isinstance(dataframe, pd.DataFrame):
        raise TypeError(f"{nombre_tabla} no es un DataFrame válido")
    if dataframe.empty:
        print(f"Aviso: {nombre_tabla} está vacío y no se cargará.")

try:
    with engine.begin() as conn:
        conn.execute(text("DROP TABLE IF EXISTS ventas_clean, ventas_error, kpi_fintech CASCADE;"))
        for nombre_tabla, dataframe in tablas_carga.items():
            if dataframe.empty:
                continue
            dataframe.to_sql(
                nombre_tabla,
                conn,
                if_exists="append",
                index=False,
                method="multi",
                chunksize=1000,
            )
    logging.info("Datos cargados correctamente en PostgreSQL")
    print("Datos cargados correctamente en PostgreSQL")
except Exception as e:
    logging.error(f"Error cargando datos en PostgreSQL: {e}")
    print("Error cargando datos en PostgreSQL:", e)

## Verificación de la carga

Consultamos la base para confirmar cuántos registros quedaron en cada tabla.

In [ ]:
with engine.connect() as conn:
    for nombre_tabla, dataframe in tablas_carga.items():
        total = conn.execute(text(f"SELECT COUNT(*) FROM {nombre_tabla};")).scalar()
        esperado = len(dataframe)
        print(f"Total registros en {nombre_tabla}: {total} | esperado: {esperado}")
        if total != esperado:
            raise ValueError(f"Validación fallida para {nombre_tabla}: cargados {total}, esperados {esperado}")